In [1]:
import pandas as pd 
import numpy as np

data_final = pd.read_csv('final_clean_V2.csv')
data_final

,filename,safety_category,safety_reason,clip_description
0,photo_150690-05-05-2024_23-29-07_jpg.rf.e53972...,Revealing_Clothing,The image shows a person wearing shorts that r...,A person wearing shorts with their legs visible.
1,train3-1046-_jpg.rf.c317d6134df09fbf126cb927a1...,Explicit_Adult,Adult Content,A woman with a tattoo on her back is engaged i...
2,Image_53_jpg.rf.71762d7cf1334bfc9b45ceb956160c...,Revealing_Clothing,The image shows a person wearing tight-fitting...,A person wearing tight-fitting jeans with a ha...
3,31897922_jpg.rf.6d5794d568be68f08f605a633dcfdf...,Revealing_Clothing,The individual is wearing a revealing outfit.,A person in revealing clothing sits on a woode...
4,87665_mp4-0014_jpg.rf.7d3440fec72f53c6688fb3d7...,Safe_Child_Friendly,No explicit content,A cartoon dog and cat interact with a woman on...
...,...,...,...,...
21335,0000001732_jpg.rf.3fbf0cf09c164fca05fadf1a07ff...,Violence_Scary,"The image contains a firearm, which is a poten...",The image shows a hand holding a Sig Sauer P22...
21336,0000002996_jpg.rf.28099ba3d4dbdd3cbe6acc69ca3c...,Violence_Scary,The presence of a firearm and the act of aimin...,"A person is holding a rifle with a scope, stan..."
21337,armas-2228-_jpg.rf.7d236e1de0c7353d666a573b004...,Violence_Scary,"The image contains a firearm, which is a tool ...",The image shows a black revolver with a wooden...
21338,0000000538_jpg.rf.c7869fa0a1c8e1602d7fd989df91...,Violence_Scary,"The image contains a firearm, which is a symbo...",The image shows a black handgun with a red and...


In [2]:
Revailing_Clothing=data_final[data_final['safety_category']=='Revealing_Clothing']
Revailing_Clothing.reset_index(drop=True, inplace=True)
len(Revailing_Clothing)

3650

In [3]:
Violence_Scary=data_final[data_final['safety_category']=='Violence_Scary']
Violence_Scary.reset_index(drop=True, inplace=True)
len(Violence_Scary)

7157

In [18]:
sports_data=pd.read_csv('data/data.csv')
sports_data['image_path']=sports_data['image_path'].apply(lambda x: x.strip('../input/'))

In [19]:
x=sports_data['image_path'].apply(lambda x: x.strip('data/'))
sports_data['target']=x.apply(lambda x: x.rsplit('/',1)[0])

sports_data.value_counts('target').keys()

# dict={}
# for i in sports_data.value_counts('target').keys():
#     dict[i]=100 
# print(dict)

Index(['badminton', 'football', 'baseball', 'volleyball', 'ennis',
       'ice_hockey', 'gymnastics', 'ble_tennis', 'swimming', 'boxing',
       'motogp', 'wwe', 'formula1', 'cricket', 'fencing', 'wrestling',
       'weight_lifting', 'hockey', 'shooting', 'basketball', 'chess',
       'kabaddi'],
      dtype='object', name='target')

In [ ]:
body_centric_sports = [
    'swimming', 'wrestling', 'wwe', 'gymnastics', 
    'kabaddi', 'boxing', 'weight_lifting'
]

general_sports = [
    # --- General / Tools  ---
    'chess', 'formula1', 'motogp', 'cricket', 'baseball', 
    'football', 'hockey', 'ice_hockey', 'volleyball', 
    'basketball', 'tennis', 'ennis', 'ble_tennis', 'badminton', 
    'fencing', 'shooting' 
]
ideal_distribution = {
    # --- Body Centric  ---
    'swimming': 350, 'wrestling': 300, 'wwe': 300, 
    'gymnastics': 300, 'kabaddi': 150, 'boxing': 200, 
    'weight_lifting': 200,

    # --- General / Tools  ---
    'fencing': 150, 'shooting': 150,
    'volleyball': 150, 'basketball': 100, 
    'ennis': 100, 'ble_tennis': 50, 'badminton': 50,
    
    # --- Low Priority  ---
    'football': 80, 'ice_hockey': 60, 'hockey': 60,
    'baseball': 50, 'cricket': 50, 'motogp': 50, 
    'formula1': 50, 'chess': 50
}

def sample_sports_smartly(source_df, distribution, label_col='label'):
    sampled_dfs = []
    
    for category, target_count in distribution.items():
        subset = source_df[source_df[label_col] == category]
        
        if len(subset) == 0:
            continue
            
        final_count = min(len(subset), target_count)
        sample = subset.sample(n=final_count, random_state=42).copy()
        
        if category in body_centric_sports:
            sample['new_label'] = 'Safe_Contextual_Body'
            type_msg = "(Body Context)"
        else:
            sample['new_label'] = 'Safe_General'  
            type_msg = "(General Safe)"
            
        sampled_dfs.append(sample)
        print(f" {category.ljust(15)}: {final_count} IMAGE -> {sample['new_label'].iloc[0]}")

    final_df = pd.concat(sampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print("="*40)
    print(final_df['new_label'].value_counts())
    print("="*40)
    
    return final_df

final_dataset = sample_sports_smartly(sports_data, ideal_distribution, label_col='target')

📊 جاري سحب العينات وتصنيفها...
✅ swimming       : 350 صور -> Safe_Contextual_Body
✅ wrestling      : 300 صور -> Safe_Contextual_Body
✅ wwe            : 300 صور -> Safe_Contextual_Body
✅ gymnastics     : 300 صور -> Safe_Contextual_Body
✅ kabaddi        : 150 صور -> Safe_Contextual_Body
✅ boxing         : 200 صور -> Safe_Contextual_Body
✅ weight_lifting : 200 صور -> Safe_Contextual_Body
✅ fencing        : 150 صور -> Safe_General
✅ shooting       : 150 صور -> Safe_General
✅ volleyball     : 150 صور -> Safe_General
✅ basketball     : 100 صور -> Safe_General
✅ ennis          : 100 صور -> Safe_General
✅ ble_tennis     : 50 صور -> Safe_General
✅ badminton      : 50 صور -> Safe_General
✅ football       : 80 صور -> Safe_General
✅ ice_hockey     : 60 صور -> Safe_General
✅ hockey         : 60 صور -> Safe_General
✅ baseball       : 50 صور -> Safe_General
✅ cricket        : 50 صور -> Safe_General
✅ motogp         : 50 صور -> Safe_General
✅ formula1       : 50 صور -> Safe_General
✅ chess          : 

In [21]:
final_dataset['new_label'].value_counts()
final_dataset.to_csv('final_clean_V3.csv', index=False)

In [48]:
pd.read_csv('final_clean_V2.csv')['safety_category'].value_counts()

safety_category
Violence_Scary           7157
Safe_Child_Friendly      4950
Explicit_Adult           4331
Revealing_Clothing       3650
Other                     803
Cartoon_With_Innuendo     211
Name: count, dtype: int64

In [47]:
RV_V_dataset=pd.read_csv('final_clean_V2.csv')[['filename','safety_category']].where(lambda x: x['safety_category'].isin(['Revealing_Clothing','Violence_Scary'])).dropna().reset_index(drop=True)
RV_V_dataset['safety_category'].value_counts()

safety_category
Violence_Scary        7157
Revealing_Clothing    3650
Name: count, dtype: int64

In [ ]:
import torch
import gc
import pandas as pd
import os
import json
import re
from tqdm.auto import tqdm
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info


print(" preparing Qwen2-VL-7B model...")

device = "cuda" if torch.cuda.is_available() else "cpu"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model_id = "Qwen/Qwen2-VL-7B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    model_id, 
    min_pixels=256*28*28, 
    max_pixels=1024*28*28  
)

print(" model loaded successfully!")


⚙️ جاري إعداد الموديل Qwen2-VL-7B...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✅ تم تحميل الموديل بنجاح!


In [ ]:
IMG_FOLDER = r"D:/projects/data/train"    

OUTPUT_CSV = "audit_results_checkpoint.csv"

TARGET_CATEGORIES = ['Violence_Scary', 'Revealing_Clothing']

def extract_json(text):
    """Extract JSON from text string."""
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        return None
    except:
        return None

df_audit = df_full[df_full['safety_category'].isin(TARGET_CATEGORIES)].copy()
print(f"total images to audit: {len(df_audit)}")
processed_files = set()
if os.path.exists(OUTPUT_CSV):
    df_done = pd.read_csv(OUTPUT_CSV)
    processed_files = set(df_done['filename'].unique())
    print(f" total processed files: {len(processed_files)}")
else:
    pd.DataFrame(columns=[
        'filename', 'old_label', 'new_label', 'is_safe', 'reason', 'clip_description'
    ]).to_csv(OUTPUT_CSV, index=False)
    print(" beginning new audit process...")

df_todo = df_audit[~df_audit['filename'].isin(processed_files)]
print(f"images remaining for processing: {len(df_todo)}")

SYSTEM_PROMPT = """You are an expert AI Safety Auditor.
The input image has a TENTATIVE old label that might be WRONG.
Your task is to re-classify it strictly based on visual context.

**RULES:**
1. **Safe_Contextual_Body**: Use this for sports (swimming, wrestling, gymnastics, bodybuilding), shirtless men in public/work, or medical contexts. Skin exposure here is SAFE.
2. **Safe_General**: Use this for ordinary safe actions, cartoons (non-violent), or fully clothed people.
3. **Unsafe_Sexual**: Use this ONLY for lingerie, underwear, seductive poses, close-ups on private parts, or sexual acts.
4. **Unsafe_Violence**: Use this for REAL blood, gore, severe injury, or threatening weapons. (Cinematic/Cartoon action without gore is usually Safe_General or a mild category).

**OUTPUT FORMAT**: Return ONLY a JSON object:
{"new_label": "Category_Name", "reason": "Short explanation", "description": "Brief visual description"}
"""

batch_results = []
SAVE_EVERY = 10  

for index, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Auditing"):
    img_path = os.path.join(IMG_FOLDER, row['filename'])
    old_label = row['safety_category']
    
    if not os.path.exists(img_path):
        continue

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": f"{SYSTEM_PROMPT}\n\nOld Label: {old_label}. Verify and correct if needed."}
            ],
        }
    ]

    try:
        text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(
            images=[img_path], 
            text=[text_prompt], 
            padding=True, 
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=200)
            
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        result_json = extract_json(output_text)
        
        if not result_json:
            result_json = {
                "new_label": "Check_Manually",
                "reason": output_text,
                "description": "Error parsing JSON"
            }

        is_safe_flag = result_json.get("new_label") in ['Safe_Contextual_Body', 'Safe_General']

        batch_results.append({
            'filename': row['filename'],
            'old_label': old_label,
            'new_label': result_json.get("new_label", "Unknown"),
            'is_safe': is_safe_flag,
            'reason': result_json.get("reason", ""),
            'clip_description': result_json.get("description", "") 
        })

    except Exception as e:
        print(f" Error processing {row['filename']}: {e}")
        torch.cuda.empty_cache()

    if len(batch_results) >= SAVE_EVERY:
        pd.DataFrame(batch_results).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
        batch_results = []

        gc.collect()
        torch.cuda.empty_cache()

if batch_results:
    pd.DataFrame(batch_results).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)



📊 إجمالي الصور المستهدفة للمراجعة: 10807
🔄 تم العثور على تشيك بوينت: 15 صورة تم معالجتها سابقاً.
🚀 المتبقي للمعالجة: 10792 صورة.


Auditing:   0%|          | 0/10792 [00:00<?, ?it/s]

🎉 تمت عملية التدقيق بنجاح!


In [ ]:



INPUT_SPORTS_CSV = "sports_subset_3k.csv"       
OUTPUT_SPORTS_CAPTIONED = "sports_captioned_checkpoint.csv"


df_sports = pd.read_csv(INPUT_SPORTS_CSV)
print(f"working on {len(df_sports)} sports images...")




if os.path.exists(OUTPUT_SPORTS_CAPTIONED):
    df_done = pd.read_csv(OUTPUT_SPORTS_CAPTIONED)
    
    processed_files = set(df_done['image_path'].unique())
    print(f"skipping {len(processed_files)} images that have already been captioned.")
    
    df_todo = df_sports[~df_sports['image_path'].isin(processed_files)]
else:
    
    pd.DataFrame(columns=[
        'image_path', 'target', 'final_label', 'clip_description', 'is_safe'
    ]).to_csv(OUTPUT_SPORTS_CAPTIONED, index=False)
    df_todo = df_sports






CAPTION_PROMPT = """Describe this image in detail.
Focus on:
1. The activity (e.g., swimming, wrestling, fencing).
2. The clothing (e.g., protective gear, swimwear, sportswear).
3. The setting (e.g., pool, strip, gym).

Use neutral, descriptive language.
If it shows skin/body in a sports context, describe it as "athlete" or "sports context".
Return JSON ONLY: {"description": "your description here"}
"""

batch_results = []


for index, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Captioning Sports"):
    
    
    
    img_path = os.path.join("data/", row['image_path'])
    
    current_label = row['new_label'] 
    sport_target = row['target']   

    if not os.path.exists(img_path):

        continue

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": CAPTION_PROMPT}
            ],
        }
    ]

    try:
        text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(images=[img_path], text=[text_prompt], padding=True, return_tensors="pt").to(device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=150)
            
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

       
        import re 
        match = re.search(r'\{.*\}', output_text, re.DOTALL)
        if match:
            try:
                res = json.loads(match.group())
                description = res.get("description", output_text)
            except:
                description = output_text
        else:
            description = output_text 

        batch_results.append({
            'image_path': row['image_path'],   
            'target': sport_target,           
            'final_label': current_label,      
            'clip_description': description,   
            'is_safe': True                    
        })

    except Exception as e:
        print(f" Error with {img_path}: {e}")
        torch.cuda.empty_cache()

    if len(batch_results) >= 10:
        pd.DataFrame(batch_results).to_csv(OUTPUT_SPORTS_CAPTIONED, mode='a', header=False, index=False)
        batch_results = []
        

if batch_results:
    pd.DataFrame(batch_results).to_csv(OUTPUT_SPORTS_CAPTIONED, mode='a', header=False, index=False)



📊 جاري العمل على 3000 صورة رياضية...
🔄 تم تخطي 2630 صورة تم وصفها سابقاً.


Captioning Sports:   0%|          | 0/370 [00:00<?, ?it/s]

🎉 تمام يا هندسة، تمت إضافة الوصف لداتا الرياضة!


In [ ]:
import pandas as pd
df_sports_new = pd.read_csv('sports_subset_3K.csv')
df_audit=pd.read_csv('audit_results_checkpoint.csv')
df_v2_filtered=pd.read_csv('final_clean_V2_filtered.csv')
print(df_sports_new.columns)
print(df_audit.columns)
print(df_v2_filtered.columns)
print(df_sports_new.shape)
print(df_audit.shape)
print(df_v2_filtered.shape)

Index(['image_path', 'target', 'final_label', 'clip_description', 'is_safe'], dtype='object')
Index(['filename', 'reason', 'clip_description', 'final_label'], dtype='object')
Index(['filename', 'safety_category', 'safety_reason', 'clip_description'], dtype='object')
(3000, 5)
(10807, 4)
(9492, 4)


In [21]:
df_v2_filtered['safety_category'].value_counts()

safety_category
Safe_Child_Friendly      4950
Explicit_Adult           4331
Cartoon_With_Innuendo     211
Name: count, dtype: int64

In [22]:
df_audit['final_label'].value_counts()

final_label
Safe_General            5126
Unsafe_Violence         4599
Safe_Contextual_Body    1020
Unsafe_Sexual             62
Name: count, dtype: int64

In [23]:
df_sports_new['final_label'].value_counts()

final_label
Safe_Contextual_Body    1800
Safe_General            1200
Name: count, dtype: int64

In [25]:
df_audit.drop(['old_label','new_label','is_safe'], axis=1, inplace=True)

In [27]:
df_audit.to_csv('audit_results_checkpoint.csv', index=False)

In [33]:
df_sports_new.drop(['target','is_safe'], axis=1, inplace=True)

In [58]:
print(df_sports_new.columns)
print(df_audit.columns)
print(df_v2_filtered.columns)

Index(['final_label', 'clip_description', 'filename'], dtype='object')
Index(['filename', 'clip_description', 'final_label'], dtype='object')
Index(['filename', 'clip_description', 'final_label'], dtype='object')


In [37]:
df_v2_filtered['final_label']=df_v2_filtered['safety_category']
df_v2_filtered.drop(['safety_category'], axis=1, inplace=True)
df_v2_filtered.columns

Index(['filename', 'safety_reason', 'clip_description', 'final_label'], dtype='object')

In [39]:
df_v2_filtered.drop(['safety_reason'], axis=1, inplace=True)
df_audit.drop(['reason'], axis=1, inplace=True)

In [ ]:
df_audit['filename'] = df_audit['filename'].apply(lambda x: f"data/train/{x}")
df_audit

,filename,clip_description,final_label
0,data/train/photo_150690-05-05-2024_23-29-07_jp...,The image depicts a person in a public setting...,Safe_General
1,data/train/Image_53_jpg.rf.71762d7cf1334bfc9b4...,The image depicts a person wearing tight-fitti...,Safe_General
2,data/train/31897922_jpg.rf.6d5794d568be68f08f6...,"The person is wearing a blue top, a white jack...",Safe_General
3,data/train/breast_2922915221383594468_jpeg.rf....,A person wearing a green T-shirt and jeans sta...,Safe_General
4,data/train/7647297_jpg.rf.0364dc3a8c2befa56827...,A person in a white dress stands with their ba...,Safe_General
...,...,...,...
10802,data/train/0000001732_jpg.rf.3fbf0cf09c164fca0...,A hand is holding a handgun with a silver and ...,Safe_General
10803,data/train/0000002996_jpg.rf.28099ba3d4dbdd3cb...,A person is holding a rifle on a tripod in an ...,Safe_General
10804,data/train/armas-2228-_jpg.rf.7d236e1de0c7353d...,"The image shows a revolver, a type of firearm,...",Unsafe_Violence
10805,data/train/0000000538_jpg.rf.c7869fa0a1c8e1602...,The image shows a gun against a red background...,Unsafe_Violence


In [44]:
df_v2_filtered['filename'] = df_audit['filename'].apply(lambda x: f"data/train/{x}")
df_v2_filtered

,filename,clip_description,final_label
0,data/train/data/train/photo_150690-05-05-2024_...,A woman with a tattoo on her back is engaged i...,Explicit_Adult
1,data/train/data/train/Image_53_jpg.rf.71762d7c...,A cartoon dog and cat interact with a woman on...,Safe_Child_Friendly
2,data/train/data/train/31897922_jpg.rf.6d5794d5...,"A woman with long hair is topless, looking at ...",Explicit_Adult
3,data/train/data/train/breast_29229152213835944...,A group of people standing in a circle with th...,Safe_Child_Friendly
4,data/train/data/train/7647297_jpg.rf.0364dc3a8...,A person in revealing clothing poses for a photo.,Explicit_Adult
...,...,...,...
9487,data/train/data/train/armas-808-_jpg.rf.7d8421...,"The image shows two foxes in a forest setting,...",Explicit_Adult
9488,data/train/data/train/armas-803-_jpg.rf.3bebef...,The image shows two animated cats in a sexual ...,Explicit_Adult
9489,data/train/data/train/0000002935_jpg.rf.cee06f...,The image depicts two animated characters in a...,Explicit_Adult
9490,data/train/data/train/armas-1867-_jpg.rf.3c6ff...,The visual shows two animated characters engag...,Explicit_Adult


In [56]:
df_sports_new['filename']=df_sports_new['image_path']
df_sports_new.drop(['image_path'], axis=1, inplace=True)

In [57]:
df_sports_new

,final_label,clip_description,filename
0,Safe_General,A group of individuals are engaged in a fencin...,data/fencing/00000439.JPG
1,Safe_Contextual_Body,A gymnast is preparing for a routine on the ba...,data/gymnastics/00000276.jpg
2,Safe_General,The image depicts a group of individuals engag...,data/fencing/00000211.jpg
3,Safe_Contextual_Body,"{\n ""description"": ""The image shows two separ...",data/swimming/00000620.jpg
4,Safe_General,The image depicts an outdoor tennis court. The...,data/badminton/00000546.jpg
...,...,...,...
2995,Safe_Contextual_Body,An athlete is performing a weightlifting exerc...,data/weight_lifting/00000498.jpg
2996,Safe_Contextual_Body,An athlete is performing a handstand on a blue...,data/gymnastics/00000581.jpg
2997,Safe_Contextual_Body,An athlete is performing a handstand on a bala...,data/gymnastics/00000518.jpg
2998,Safe_Contextual_Body,The image depicts a dynamic scene from a kabad...,data/kabaddi/00000275.jpg


In [ ]:

df_final_v2 = pd.concat([df_sports_new, df_audit, df_v2_filtered], axis=0, ignore_index=True)

df_final_v2 = df_final_v2[['filename', 'final_label', 'clip_description']]


df_final_v2 = df_final_v2.sample(frac=1, random_state=42).reset_index(drop=True)

initial_len = len(df_final_v2)
df_final_v2 = df_final_v2.drop_duplicates(subset=['filename'])
if len(df_final_v2) < initial_len:
    print(f"deleted {initial_len - len(df_final_v2)} duplicated images  .")

print("="*40)
print("Final Dataset Statistics:")
print(f"Total Images: {len(df_final_v2)}")
print("-" * 20)
print(df_final_v2['final_label'].value_counts())
print("="*40)

df_final_v2.to_csv("final_dataset_v2_ready.csv", index=False)
print("Dataset saved as 'final_dataset_v2_ready.csv'")

🎉 Final Dataset Statistics:
Total Images: 23299
--------------------
final_label
Safe_General             6326
Safe_Child_Friendly      4950
Unsafe_Violence          4599
Explicit_Adult           4331
Safe_Contextual_Body     2820
Cartoon_With_Innuendo     211
Unsafe_Sexual              62
Name: count, dtype: int64
✅ Dataset saved as 'final_dataset_v2_ready.csv'


In [ ]:
df_current = pd.read_csv("final_dataset_v2_ready.csv")


label_map = {
    'Safe_Child_Friendly': 'Safe_General',     
    'Explicit_Adult': 'Unsafe_Sexual',          
    'Cartoon_With_Innuendo': 'Unsafe_Sexual',   
    'Safe_General': 'Safe_General',
    'Unsafe_Violence': 'Unsafe_Violence',
    'Unsafe_Sexual': 'Unsafe_Sexual',
    'Safe_Contextual_Body': 'Safe_Contextual_Body'
}

df_current['final_label'] = df_current['final_label'].map(label_map)
df_current.to_csv("final_dataset_v2_ready.csv", index=False)

print(" Dataset saved as 'final_dataset_v2_ready.csv'")


✅ Dataset saved as 'final_dataset_v2_ready.csv'


In [61]:
df_current['final_label'].value_counts()

final_label
Safe_General            11276
Unsafe_Sexual            4604
Unsafe_Violence          4599
Safe_Contextual_Body     2820
Name: count, dtype: int64

In [ ]:
df_safe_gen = df_current[df_current['final_label'] == 'Safe_General']
df_safe_gen_balanced = df_safe_gen.sample(n=5000, random_state=42)

df_others = df_current[df_current['final_label'] != 'Safe_General']

df_balanced_final = pd.concat([df_safe_gen_balanced, df_others])
df_balanced_final = df_balanced_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("="*40)
print("Final Balanced Distribution (V2):")
print("-" * 20)
print(df_balanced_final['final_label'].value_counts())
print("-" * 20)
print(f"Total Images: {len(df_balanced_final)}")
print("="*40)

df_balanced_final.to_csv("dataset_v2_gold.csv", index=False)

🏆 Final Balanced Distribution (V2):
--------------------
final_label
Safe_General            5000
Unsafe_Sexual           4604
Unsafe_Violence         4599
Safe_Contextual_Body    2820
Name: count, dtype: int64
--------------------
Total Images: 17023


In [ ]:

df_final_v2 = pd.concat([df_audit, df_v2_filtered], axis=0, ignore_index=True)

df_final_v2 = df_final_v2[['filename', 'final_label', 'clip_description']]


df_final_v2 = df_final_v2.sample(frac=1, random_state=42).reset_index(drop=True)

initial_len = len(df_final_v2)
df_final_v2 = df_final_v2.drop_duplicates(subset=['filename'])
if len(df_final_v2) < initial_len:
    print(f" Deleted {initial_len - len(df_final_v2)} duplicate images.")

print("="*40)
print("Final Dataset Statistics:")
print(f"Total Images: {len(df_final_v2)}")
print("-" * 20)
print(df_final_v2['final_label'].value_counts())
print("="*40)

df_final_v2.to_csv("final_dataset_v2_ready.csv", index=False)
print(" Dataset saved as 'final_dataset_v2_ready.csv'")

In [ ]:
df_safe_gen = df_current[df_current['final_label'] == 'Safe_General']
df_safe_gen_balanced = df_safe_gen.sample(n=5000, random_state=42)

df_others = df_current[df_current['final_label'] != 'Safe_General']

df_balanced_final = pd.concat([df_safe_gen_balanced, df_others])
df_balanced_final = df_balanced_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("="*40)
print(" Final Balanced Distribution (V2):")
print("-" * 20)
print(df_balanced_final['final_label'].value_counts())
print("-" * 20)
print(f"Total Images: {len(df_balanced_final)}")
print("="*40)

df_balanced_final.to_csv("dataset_v2_gold.csv", index=False)